# A/B Testing

A/B testing is a **randomised controlled experiment** comparing two versions (A=control, B=treatment) of a product, webpage, or intervention. The goal is to determine whether a change causes a measurable improvement in a chosen metric.

**End-to-end workflow:**
1. Define the metric and minimum detectable effect (MDE)
2. Calculate required sample size (power analysis)
3. Run the experiment and collect data
4. Analyse results (hypothesis test + effect size + CI)
5. Make a decision

## Step 1 & 2 — Define Metric and Calculate Sample Size

We're testing whether a new checkout button (green vs grey) increases conversion rate.
- Baseline conversion: $p_0 = 0.05$ (5%)
- Minimum Detectable Effect: $\delta = 0.01$ (we want to detect 1 pp lift)
- $\alpha = 0.05$, power = 0.80

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower

rng = np.random.default_rng(seed=42)

p0    = 0.05    # baseline conversion
p1    = 0.06    # expected conversion under treatment (MDE = 1 pp)
alpha = 0.05
power = 0.80

# Effect size (Cohen's h for proportions)
h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p0))

analysis = NormalIndPower()
n_per_group = analysis.solve_power(effect_size=h, alpha=alpha, power=power, alternative='two-sided')

print('A/B Test Sample Size Calculation')
print(f'  Baseline conversion p0   : {p0:.0%}')
print(f'  Target conversion p1     : {p1:.0%}')
print(f'  Minimum Detectable Effect: {p1-p0:+.0%} ({(p1-p0)/p0:.0%} relative lift)')
print(f'  Cohen\'s h                : {h:.4f}')
print(f'  Required n per group     : {int(np.ceil(n_per_group)):,}')
print(f'  Total sample size        : {int(np.ceil(n_per_group))*2:,}')


A/B Test Sample Size Calculation
  Baseline conversion p0   : 5%
  Target conversion p1     : 6%
  Minimum Detectable Effect: +1% (20% relative lift)
  Cohen's h                : 0.0439
  Required n per group     : 8,143
  Total sample size        : 16,286


## Step 3 — Run the Simulated Experiment

In [2]:
n = int(np.ceil(n_per_group))

# True treatment effect: p_B = 0.063 (slightly larger than MDE — realistic)
true_p_A = 0.05
true_p_B = 0.063

conversions_A = rng.binomial(1, true_p_A, size=n)
conversions_B = rng.binomial(1, true_p_B, size=n)

obs_A = conversions_A.sum()
obs_B = conversions_B.sum()

print(f'Control  (A): {obs_A}/{n} conversions  ({obs_A/n:.2%})')
print(f'Treatment(B): {obs_B}/{n} conversions  ({obs_B/n:.2%})')
print(f'Observed lift: {(obs_B/n - obs_A/n):+.2%}')


Control  (A): 405/8143 conversions  (4.97%)
Treatment(B): 517/8143 conversions  (6.35%)
Observed lift: +1.38%


## Step 4 — Analyse Results

In [3]:
# Two-proportion z-test
counts  = np.array([obs_B, obs_A])
nobs    = np.array([n, n])
z_stat, p_val = proportions_ztest(counts, nobs, alternative='two-sided')

# 95% CI for each proportion (Wilson)
ci_A = proportion_confint(obs_A, n, alpha=0.05, method='wilson')
ci_B = proportion_confint(obs_B, n, alpha=0.05, method='wilson')

# 95% CI for the difference
diff      = obs_B/n - obs_A/n
se_diff   = np.sqrt(obs_A/n*(1-obs_A/n)/n + obs_B/n*(1-obs_B/n)/n)
z_crit    = stats.norm.ppf(0.975)
diff_lo   = diff - z_crit * se_diff
diff_hi   = diff + z_crit * se_diff

# Cohen's h (effect size for proportions)
h_obs = 2*np.arcsin(np.sqrt(obs_B/n)) - 2*np.arcsin(np.sqrt(obs_A/n))

print('A/B Test Results')
print('=' * 50)
print(f'Control   A: {obs_A/n:.4f}  95% CI: ({ci_A[0]:.4f}, {ci_A[1]:.4f})')
print(f'Treatment B: {obs_B/n:.4f}  95% CI: ({ci_B[0]:.4f}, {ci_B[1]:.4f})')
print(f'Difference : {diff:+.4f}  95% CI: ({diff_lo:+.4f}, {diff_hi:+.4f})')
print(f'z-statistic: {z_stat:.4f}')
print(f'p-value    : {p_val:.4f}')
print(f"Cohen's h  : {h_obs:.4f}")
print('=' * 50)
if p_val < 0.05:
    print('Statistically significant at alpha=0.05 → Ship treatment B.')
else:
    print('Not significant → Do not ship treatment B (insufficient evidence).')
print()
print('Practical significance check:')
print(f'  Observed lift {diff:+.2%} vs MDE {p1-p0:+.2%}')
if abs(diff) >= (p1-p0):
    print('  Lift meets or exceeds MDE — practically significant.')
else:
    print('  Lift is below MDE — may not be worth the engineering cost.')


A/B Test Results
Control   A: 0.0497  95% CI: (0.0452, 0.0547)
Treatment B: 0.0635  95% CI: (0.0584, 0.0690)
Difference : +0.0138  95% CI: (+0.0067, +0.0208)
z-statistic: 3.7976
p-value    : 0.0001
Cohen's h  : 0.0596
Statistically significant at alpha=0.05 → Ship treatment B.

Practical significance check:
  Observed lift +1.38% vs MDE +1.00%
  Lift meets or exceeds MDE — practically significant.


## Common A/B Testing Pitfalls

1. **Peeking:** stopping as soon as p < 0.05 inflates Type I error — always pre-register your sample size.
2. **Multiple metrics:** testing 10 metrics at once → at least one will be 'significant' by chance. Apply FDR correction.
3. **Novelty effect:** users interact differently with new features initially — run the test long enough for novelty to wear off.
4. **Network effects:** for social features, treated and control users interact → standard A/B assumptions break down (use cluster randomisation).
5. **Segment imbalance:** ensure randomisation produces balanced groups on key covariates (age, device type, geography).

In [4]:
# Peeking simulation: what happens if you stop whenever p < 0.05?
n_total  = 2000
n_sims   = 5000
false_pos_peeking  = 0
false_pos_fixed    = 0

for _ in range(n_sims):
    # Both groups same conversion rate (H0 true)
    A = rng.binomial(1, 0.05, size=n_total)
    B = rng.binomial(1, 0.05, size=n_total)

    # Fixed sample test
    _, p = proportions_ztest([B.sum(), A.sum()], [n_total, n_total])
    if p < 0.05:
        false_pos_fixed += 1

    # Peeking: check p-value every 100 observations
    peeked = False
    for check in range(100, n_total+1, 100):
        _, p_peek = proportions_ztest([B[:check].sum(), A[:check].sum()], [check, check])
        if p_peek < 0.05:
            peeked = True
            break
    if peeked:
        false_pos_peeking += 1

print(f'False positive rate (fixed sample) : {false_pos_fixed/n_sims:.3f}  (expected ~0.05)')
print(f'False positive rate (peeking)      : {false_pos_peeking/n_sims:.3f}  (inflated!)')
print()
print('Peeking can triple or quadruple your false positive rate.')


False positive rate (fixed sample) : 0.051  (expected ~0.05)
False positive rate (peeking)      : 0.244  (inflated!)

Peeking can triple or quadruple your false positive rate.


---
## ML/AI Connection

- **Every major tech product** (Google, Meta, Netflix, Airbnb) runs thousands of A/B tests simultaneously — the statistical engine is exactly the proportion z-test implemented here.
- **Online learning / multi-armed bandits** (Thompson Sampling, UCB) are adaptive alternatives to fixed A/B tests — they allocate more traffic to the winning arm during the experiment.
- **Interleaving experiments** (Netflix) compare ranking algorithms by interleaving both lists to a single user — more sensitive than between-user A/B tests.
